In [0]:
import yaml
import os
import shutil

config_path = "config.yaml" if os.path.exists("config.yaml") else "../config.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

CATALOG = config["storage"]["catalog_name"]
SCHEMA = config["storage"]["schema_name"]
VOLUME = config["storage"]["volume_name"]
BATCH_SIZE = config["pipeline"]["batch_size"]
HF_DATASET = config["dataset"]["hf_name"]

BATCH_DIR = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/current_batch"

# 1. Sprawdzamy, którą paczkę właśnie skończyliśmy przetwarzać
state_row = spark.sql(f"""
    SELECT last_batch_id, last_offset 
    FROM {CATALOG}.{SCHEMA}.pipeline_state 
    WHERE dataset_name = '{HF_DATASET}'
    ORDER BY last_batch_id DESC LIMIT 1
""").collect()[0]

completed_batch_id = state_row["last_batch_id"] + 1
new_offset = state_row["last_offset"] + BATCH_SIZE

# 2. Usuwamy fizyczne pliki .jpg z bieżącej paczki w Volume
removed_files = 0
freed_bytes = 0

if os.path.exists(BATCH_DIR):
    for fname in os.listdir(BATCH_DIR):
        fpath = os.path.join(BATCH_DIR, fname)
        if os.path.isfile(fpath):
            freed_bytes += os.path.getsize(fpath)
            os.remove(fpath)
            removed_files += 1

# 3. Czyścimy tabelę tymczasową silver_current_batch (wektory z tej paczki już nauczyły modele)
if spark.catalog.tableExists(f"{CATALOG}.{SCHEMA}.silver_current_batch"):
    spark.sql(f"TRUNCATE TABLE {CATALOG}.{SCHEMA}.silver_current_batch")

print(f"Wyczyszczono Paczkę #{completed_batch_id}: usunięto {removed_files} plików .jpg ({freed_bytes / 1024:.1f} KB).")

In [0]:
# Zapisujemy w pipeline_state, że Paczka #completed_batch_id jest gotowa i następna startuje od new_offset
spark.sql(f"""
    INSERT INTO {CATALOG}.{SCHEMA}.pipeline_state
    VALUES ('{HF_DATASET}', {completed_batch_id}, {new_offset}, current_timestamp())
""")

print(f"Zaktualizowano pipeline_state! Kolejny przebieg pobierze Paczkę #{completed_batch_id + 1} (od zdjęcia nr {new_offset}).")

# Podgląd historii stanów pipeline'u
display(
    spark.sql(f"""
        SELECT * FROM {CATALOG}.{SCHEMA}.pipeline_state 
        WHERE dataset_name = '{HF_DATASET}'
        ORDER BY last_batch_id DESC
    """)
)